# Baseline Reproduction: EuroSAT Dataset
**Course:** CSCI 3052U - Machine Learning I  
**Team:** Team SixtyFive.AI | Milestone 3

**Project:** Out-of-Distribution Generalization in Satellite Land Cover Classification: A Geographic Domain-Shift Approach.

This notebook implements the team's **majority-class benchmark** and **RGB PCA + Random Forest baseline** using the geographic partition identified during the EuroSAT project. The objective is to measure how classification changes between random validation within development countries and a candidate out-of-distribution (OOD) set from different countries. **Macro-F1** is the primary evaluation metric.

The implementation follows the team's existing baseline scripts now merged into `main` (`src/baseline.py` and `src/run_rgb_baseline.py`). This notebook runs the model logic itself; it does **not** load precomputed result CSVs as if they were fresh predictions. A geographic manifest CSV is required for sample-to-partition mapping, and the actual RGB image files are required to train the Random Forest. Previous team metrics appear in a clearly labelled reference comparison after new metrics are computed.

**Data source:** [EuroSAT v2 — Zenodo record 7711810](https://zenodo.org/records/7711810). Extract the RGB archive under `data/EuroSAT_RGB/` (either supported nested layout). Use the team's `outputs/geographic_manifest_rebuilt.csv` on `main` (or regenerate it with `src/build_geographic_manifest.py` if missing). Install project dependencies, then use **Kernel > Restart & Run All**. Stored reference values are not proof that this notebook has been executed. No new execution outputs have been fabricated.


## 1. Environment and reproducibility
Set up paths, dependencies, and fixed model settings before loading the dataset. The notebook works from the repository root or `notebooks/`. A reproducible run requires the same geographic manifest, RGB dataset version, dependencies, random seed, and image preprocessing.

The code uses Python packages `numpy`, `pandas`, `Pillow`, `scikit-learn`, and `matplotlib`. `requirements.txt` should be updated and version-pinned after testing the final environment. Results and figures are saved to a separate notebook-specific folder to avoid overwriting your team's prior output files.


In [ ]:
from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
IMAGE_SIZE = (16, 16)
PCA_COMPONENTS = 100
N_TREES = 200

# Works when executed from repo root or notebooks/.
CWD = Path.cwd().resolve()
if (CWD / 'outputs').is_dir():
    ROOT = CWD
elif (CWD.parent / 'outputs').is_dir():
    ROOT = CWD.parent
else:
    raise FileNotFoundError('Run inside the team project folder (or its notebooks/ directory).')

OUTPUTS = ROOT / 'outputs'
MANIFEST_PATH = OUTPUTS / 'geographic_manifest_rebuilt.csv'
print('Project root:', ROOT)
print('Python:', platform.python_version(), '| scikit-learn:', __import__('sklearn').__version__)
print('Seed:', SEED)

## 2. Geographic manifest and dataset partitions
Read the team's `geographic_manifest_rebuilt.csv`, inspect the number of samples in each candidate partition, and tabulate class counts. This is metadata needed to construct the experiment-not precomputed model results.

The original manifest distinguishes `development`, `ood_test`, `excluded_region`, and `unresolved`. Excluded and unresolved samples are not used to evaluate either model. A candidate country-level partition is **not** itself proof of complete spatial independence.


In [ ]:
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f'Missing {MANIFEST_PATH}. Copy the validated geographic_manifest_rebuilt.csv '
        'from outputs/ on main or run src/build_geographic_manifest.py first.'
    )
manifest = pd.read_csv(MANIFEST_PATH)
needed = {'sample_id', 'class', 'candidate_partition'}
assert needed.issubset(manifest.columns), f'Missing manifest columns: {needed - set(manifest.columns)}'
assert manifest['sample_id'].notna().all() and manifest['sample_id'].is_unique, 'Sample IDs must be unique.'
print('All manifest rows:', len(manifest))
print('\nPartition counts:')
print(manifest['candidate_partition'].value_counts(dropna=False).to_string())
print('\nClass counts by partition:')
display(pd.crosstab(manifest['class'], manifest['candidate_partition']))

In [ ]:
development = manifest.loc[manifest['candidate_partition'].eq('development')].copy()
ood_test = manifest.loc[manifest['candidate_partition'].eq('ood_test')].copy()
assert len(development) > 1 and len(ood_test) > 0
assert set(development['sample_id']).isdisjoint(set(ood_test['sample_id']))
if 'country_code' in manifest:
    dev_countries = set(development['country_code'].dropna())
    ood_countries = set(ood_test['country_code'].dropna())
    overlapping = dev_countries & ood_countries
    print('Shared candidate country codes:', sorted(overlapping))
    assert not overlapping, 'Country overlap in development and geographic test.'

# Same random stratified development split as the team's two baseline scripts.
dev_train, dev_val = train_test_split(
    development, test_size=0.2, random_state=SEED, stratify=development['class']
)
print(f'Development train: {len(dev_train):,} | Validation: {len(dev_val):,} | OOD: {len(ood_test):,}')

## 3. Naive benchmark: majority-class classifier
The `DummyClassifier(strategy="most_frequent")` always predicts the most frequent label in the development **training** split. It never looks at satellite image pixels. This simple benchmark provides a performance floor for comparison with the image-based model.

Evaluate the dummy model on the development validation and candidate OOD test sets. Report accuracy and Macro-F1 separately for both partitions; class distributions can affect the dummy score even though it predicts the same class everywhere.


In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(np.zeros((len(dev_train), 1)), dev_train['class'])
majority_class = dummy.classes_[dummy.class_prior_.argmax()]
print('Predicted majority class:', majority_class)

rows = []
for part_name, subset in [('development_validation', dev_val), ('ood_test', ood_test)]:
    predictions = dummy.predict(np.zeros((len(subset), 1)))
    rows.append({
        'model': 'Majority-class baseline',
        'partition': part_name,
        'n_samples': len(subset),
        'accuracy': accuracy_score(subset['class'], predictions),
        'macro_f1': f1_score(subset['class'], predictions, labels=sorted(manifest['class'].dropna().unique()), average='macro', zero_division=0),
    })
dummy_results = pd.DataFrame(rows)
display(dummy_results.round(4))

## 4. RGB image inventory and preprocessing
Index the actual EuroSAT RGB image files and match them to the geographic manifest by sample ID. Check for duplicates and missing required files rather than silently dropping images.

For the course-level baseline, decode each RGB image, resize to **16 × 16** pixels, flatten its three channels into **768 values**, and scale pixels to `[0, 1]`. The model is **RGB-only**; the multispectral archive examined during Milestone 2 is not used in this baseline.


In [ ]:
candidates = [ROOT / 'data' / 'EuroSAT_RGB' / 'EuroSAT_RGB', ROOT / 'data' / 'EuroSAT_RGB']
RGB_ROOT = next((p for p in candidates if p.is_dir() and any(c.is_dir() for c in p.iterdir())), None)
if RGB_ROOT is None:
    raise FileNotFoundError(
        'EuroSAT RGB images missing. Extract the RGB dataset under '
        'data/EuroSAT_RGB/EuroSAT_RGB/<class>/ and rerun this cell.'
    )

image_paths = {}
for folder in sorted(RGB_ROOT.iterdir()):
    if folder.is_dir():
        for img in folder.iterdir():
            if img.is_file() and img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                if img.stem in image_paths:
                    raise ValueError(f'Duplicate image sample ID: {img.stem}')
                image_paths[img.stem] = img

selected = manifest.loc[manifest['candidate_partition'].isin(['development', 'ood_test'])].copy()
missing_ids = sorted(set(selected['sample_id']) - set(image_paths))
print('RGB root:', RGB_ROOT)
print('Images indexed:', len(image_paths))
print('Images needed:', len(selected), '| Missing:', len(missing_ids))
if missing_ids:
    raise FileNotFoundError(f'Missing {len(missing_ids)} required RGB images; examples: {missing_ids[:8]}')
assert selected['sample_id'].is_unique

In [ ]:
features = []
for sample_id in selected['sample_id']:
    with Image.open(image_paths[sample_id]) as image:
        image = image.convert('RGB').resize(IMAGE_SIZE)
        features.append(np.asarray(image, dtype=np.float32).flatten() / 255.0)
X = np.asarray(features, dtype=np.float32)
y = selected['class'].to_numpy()
partitions = selected['candidate_partition'].to_numpy()
print('Features shape:', X.shape, '| Feature range:', (float(X.min()), float(X.max())))
assert X.shape == (len(selected), 16 * 16 * 3)

## 5. Train the course-level baseline: PCA + Random Forest
A scikit-learn pipeline fits **PCA with 100 components** followed by a **200-tree Random Forest** with balanced class weights and random seed **42**. PCA is fitted on the training data through the pipeline; neither validation data nor OOD test data is used to fit PCA or train the classifier.

Development data is split using random stratification (`test_size=0.2`, `random_state=42`). The OOD partition is reserved for evaluation. A country-disjoint OOD test and random within-region validation answer different questions, so interpret their scores accordingly.


In [ ]:
X_dev, y_dev = X[partitions == 'development'], y[partitions == 'development']
X_ood, y_ood = X[partitions == 'ood_test'], y[partitions == 'ood_test']
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.2, random_state=SEED, stratify=y_dev
)
assert len(X_train) == len(dev_train) and len(X_val) == len(dev_val)
# Confirm the image-feature split uses exactly the same ordered manifest rows as the dummy baseline.
assert np.array_equal(y_train, dev_train['class'].to_numpy()), 'Training labels differ from manifest split.'
assert np.array_equal(y_val, dev_val['class'].to_numpy()), 'Validation labels differ from manifest split.'

model = Pipeline([
    ('pca', PCA(n_components=PCA_COMPONENTS, random_state=SEED)),
    ('random_forest', RandomForestClassifier(
        n_estimators=N_TREES,
        random_state=SEED,
        n_jobs=-1,
        class_weight='balanced',
    )),
])
print(f'Training on {len(y_train):,} development images...')
model.fit(X_train, y_train)
print('Training complete.')

## 6. Evaluate the image-based baseline
Run the trained PCA + Random Forest model on development validation and candidate OOD images. Calculate accuracy and **Macro-F1** directly from predictions, along with per-class precision, recall, F1, and sample support.

Macro-F1 is the mean of the ten class F1 scores, so performance on underrepresented land-cover classes matters. The OOD set is used only here for evaluation, not training.


In [ ]:
results = []
class_results = []
all_classes = sorted(np.unique(y))
for part_name, X_eval, y_eval in [
    ('development_validation', X_val, y_val),
    ('ood_test', X_ood, y_ood),
]:
    predicted = model.predict(X_eval)
    results.append({
        'model': 'PCA + Random Forest (RGB)',
        'partition': part_name,
        'n_samples': len(y_eval),
        'accuracy': accuracy_score(y_eval, predicted),
        'macro_f1': f1_score(y_eval, predicted, labels=all_classes, average='macro', zero_division=0),
    })
    report = classification_report(y_eval, predicted, labels=all_classes,
                                   output_dict=True, zero_division=0)
    for class_name in all_classes:
        metrics = report[class_name]
        class_results.append({
            'model': 'PCA + Random Forest (RGB)',
            'partition': part_name,
            'class': class_name,
            'support': metrics['support'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1-score'],
        })
rf_results = pd.DataFrame(results)
per_class_results = pd.DataFrame(class_results)
display(rf_results.round(4))
display(per_class_results.round(3))

## 7. Baseline table and geographic discrepancy analysis
Compare the newly computed majority-class benchmark and PCA + Random Forest metrics in one table. For each model, calculate the geographic discrepancy as:

**Macro-F1 gap = development validation Macro-F1 − candidate OOD test Macro-F1.**

A positive gap means that performance is lower on the geographically shifted test partition. Also inspect **per-class F1 gaps** to see which land-cover categories change most. This is an observed difference between project partitions, not an asserted reproduction gap against a published paper.


In [ ]:
combined = pd.concat([dummy_results, rf_results], ignore_index=True)
summary = combined.pivot(index='model', columns='partition', values='macro_f1')
summary['f1_drop'] = summary['development_validation'] - summary['ood_test']
print('Overall reproduced baseline comparison:')
display(summary.round(4))

per_class_pivot = per_class_results.pivot(index='class', columns='partition', values='f1')
per_class_pivot['f1_drop'] = (
    per_class_pivot['development_validation'] - per_class_pivot['ood_test']
)
per_class_pivot = per_class_pivot.sort_values('f1_drop', ascending=False)
print('Per-class gap (RGB model):')
display(per_class_pivot.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
summary[['development_validation', 'ood_test']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Macro-F1 by baseline and partition')
axes[0].set_ylabel('Macro-F1')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(['Development validation', 'Geographic OOD'])
per_class_pivot['f1_drop'].plot(kind='bar', ax=axes[1])
axes[1].set_title('RGB model: per-class Macro-F1 difference')
axes[1].set_ylabel('Validation F1 − OOD F1')
axes[1].tick_params(axis='x', rotation=65)
plt.tight_layout()
plt.show()

## 8. Check reproducibility against previously saved team results
The team’s baseline experiments (now merged into `main`) produced the reference metrics below. Compare those stored numbers against **newly calculated notebook results** after the training and evaluation cells have completed.

The reference numbers below are labelled as past team outputs. They are **not used to train, predict, or calculate the new Macro-F1 values**. Nonzero differences may result from a changed image set, manifest, split, code, library versions, or preprocessing details and should be investigated rather than hidden.


In [ ]:
reference = pd.DataFrame([
    ['Majority-class baseline', 'development_validation', 3516, 0.1359499431171786, 0.0239359038557836],
    ['Majority-class baseline', 'ood_test', 6231, 0.0913176055207831, 0.016735294117647],
    ['PCA + Random Forest (RGB)', 'development_validation', 3516, 0.7084755403868032, 0.6910487683196636],
    ['PCA + Random Forest (RGB)', 'ood_test', 6231, 0.4485636334456748, 0.4364510465773787],
], columns=['model', 'partition', 'n_samples', 'accuracy', 'macro_f1'])
comparison = combined.merge(reference, on=['model', 'partition'], suffixes=('_run', '_reference'), validate='one_to_one')
comparison['macro_f1_difference'] = comparison['macro_f1_run'] - comparison['macro_f1_reference']
comparison['accuracy_difference'] = comparison['accuracy_run'] - comparison['accuracy_reference']
comparison['sample_count_difference'] = comparison['n_samples_run'] - comparison['n_samples_reference']
display(comparison.round(6))

## 9. Export measured notebook outputs
Write the notebook's freshly computed overall metrics, per-class results, geographic gap table, and comparison against earlier metrics to `outputs/milestone3_notebook/`.

The export files are outputs of the executed Python analysis. They are not input substitutes for the image data. Execute all preceding code cells successfully before exporting the results.


In [ ]:
export_dir = OUTPUTS / 'milestone3_notebook'
export_dir.mkdir(parents=True, exist_ok=True)
combined.to_csv(export_dir / 'combined_baseline_results.csv', index=False)
per_class_results.to_csv(export_dir / 'rgb_per_class_results.csv', index=False)
per_class_pivot.reset_index().to_csv(export_dir / 'rgb_f1_gap_analysis.csv', index=False)
comparison.to_csv(export_dir / 'reference_comparison.csv', index=False)
print('Saved outputs:', export_dir)

## 10. Interpretation, discrepancies, and limitations
**Main research question:** Does a land-cover classifier trained on images from the development countries generalize to a candidate test set from other countries?

**Reference finding (previous team run, not newly measured here):** The RGB PCA + Random Forest achieved Macro-F1 of **0.6910** on development validation and **0.4365** on candidate OOD test, a drop of about **0.2546**. The majority-class benchmark achieved **0.0239** and **0.0167**, respectively. The new tables in Sections 6–8 must be used to confirm whether the current run agrees.

**Discrepancy analysis:**
- A substantial positive Macro-F1 gap indicates weaker performance on the geographically different sample distribution. It does **not** establish that geography alone caused the decline; class prevalence, scene composition, seasonality, and candidate partition quality can also matter.
- The dummy classifier predicts the majority training class regardless of image pixels. Different class frequencies in the validation and OOD sets can change its accuracy and Macro-F1.
- The team's prior per-class outputs showed especially large declines for **Pasture** (about **0.5828**) and **Herbaceous Vegetation** (about **0.4588**), compared with smaller declines for **Forest** (about **0.0524**) and **Sea/Lake** (about **0.0216**). Verify those differences using the newly computed per-class table above.
- Some categories have very different representation across partitions. Low OOD support can make an individual class result less stable. Interpret per-class comparisons together with their support counts.

**Limitations and reproducibility conditions:**
- Validation is **randomly stratified within development countries**, not a held-out spatial block. Disjoint candidate country codes do not eliminate all possible patch-level spatial autocorrelation or mapping errors.
- The manifest contains `excluded_region` and `unresolved` samples; they are not evaluated. The geographic assignments should be finalized and audited before claiming fully independent geographic generalization.
- This is an **RGB** baseline, not a 13-band multispectral baseline or CNN. Those are separate potential later experiments.
- A reproducibility claim requires a successful **Restart & Run All** using the correct EuroSAT images, geographic manifest, and tested environment. The notebook has not been independently run on the full dataset in this handoff.

## 11. Start / Stop / Continue plan
**Start:** Audit possible spatial overlap; review border/unresolved samples; finalize the geographical split and verify reproducibility in a clean environment.

**Stop:** Treating random in-region validation or provisional geographic labels as proof of leakage-free generalization; relying solely on overall accuracy.

**Continue:** Fixed random seeds, PCA fitting only on training data, Macro-F1 and per-class reporting, and explicit documentation of limitations.